## Geometry Clustering (train/dev)

`outputs/physics_feature_analysis_v2/geometry_sample_features.csv`의 기하학 피처를 사용해 구조물 형태를 군집화합니다.
- 목표: 소수 shape type으로 자동 분리
- 방법: 표준화 + KMeans + 실루엣 점수 기반 k 선택


In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeClassifier, export_text

FEATURE_PATH = '../outputs/physics_feature_analysis_v2/geometry_sample_features.csv'

df = pd.read_csv(FEATURE_PATH)
feature_cols = [c for c in df.columns if c not in ['split', 'sample_id', 'label']]

X = df[feature_cols].copy()
X = X.fillna(X.median(numeric_only=True))

print('shape:', df.shape)
print('split counts:
', df['split'].value_counts())
print('label counts:
', df['label'].value_counts(dropna=False))


In [ ]:
# 1) k 탐색: 실루엣 점수
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

sil_rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    cluster = km.fit_predict(Xs)
    sil = silhouette_score(Xs, cluster)
    sil_rows.append({'k': k, 'silhouette': sil})

sil_df = pd.DataFrame(sil_rows).sort_values('k')
print(sil_df.to_string(index=False))

best_k = int(sil_df.sort_values('silhouette', ascending=False).iloc[0]['k'])
print('
Best k by silhouette =', best_k)


In [ ]:
# 2) 최적 k로 군집화
k = best_k
kmeans = KMeans(n_clusters=k, random_state=42, n_init=50)
df['shape_cluster'] = kmeans.fit_predict(Xs)

print('cluster size:')
print(df['shape_cluster'].value_counts().sort_index())

print('
cluster x label (row-normalized):')
print(pd.crosstab(df['shape_cluster'], df['label'], normalize='index').round(3))

selected = [
    'front_slenderness',
    'front_width_frac',
    'front_base_width_frac',
    'front_top_width_frac',
    'top_area_frac',
    'top_support_width_frac',
    'front_tilt',
    'collapse_margin_proxy',
]
print('
cluster centers (mean):')
print(df.groupby('shape_cluster')[selected].mean().round(3))


In [ ]:
# 3) 군집별 대표 샘플(중심과 가장 가까운 샘플)
centers = kmeans.cluster_centers_

for c in range(k):
    idx = np.where(df['shape_cluster'].values == c)[0]
    d2 = ((Xs[idx] - centers[c]) ** 2).sum(axis=1)
    nearest = idx[np.argsort(d2)[:5]]

    cols = [
        'sample_id', 'split', 'label',
        'front_slenderness', 'front_width_frac',
        'front_base_width_frac', 'front_top_width_frac',
        'top_support_width_frac', 'front_tilt'
    ]
    print(f'\n[cluster {c}] representatives')
    print(df.iloc[nearest][cols].to_string(index=False))


In [ ]:
# 4) 사람이 읽기 쉬운 규칙(해석용)
# 군집 ID를 얕은 의사결정나무로 근사하면, 형태 구분 기준선을 빠르게 볼 수 있음.
rule_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=42)
rule_tree.fit(X, df['shape_cluster'])

print(export_text(rule_tree, feature_names=feature_cols))
print('approximation acc:', round(rule_tree.score(X, df['shape_cluster']), 4))
